In [1]:
import os, copy, gc, random, time, warnings
warnings.filterwarnings("ignore")
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
from torch.utils.data import DataLoader
from transformers import AutoTokenizer, AutoModelForSequenceClassification, get_linear_schedule_with_warmup
from transformers.utils import logging as hf_logging
from sklearn.metrics import accuracy_score, precision_recall_fscore_support, f1_score, roc_auc_score

hf_logging.set_verbosity_error()
device = "cuda" if torch.cuda.is_available() else "cpu"
print("device:", device)

def set_seed(seed=42):
    random.seed(seed); np.random.seed(seed)
    torch.manual_seed(seed); torch.cuda.manual_seed_all(seed)

DATA = {"EATD": pd.read_csv("data/eatd.csv"), "DAIC-WOZ": pd.read_csv("data/daicwoz.csv")}
for k, d in DATA.items():
    print(k, len(d), "participants,", int(d["label"].sum()), "depressed")

def evaluate(y_true, y_prob, thr=0.5):
    y_pred = (np.asarray(y_prob) >= thr).astype(int)
    pc, rc, fc, _ = precision_recall_fscore_support(y_true, y_pred, average=None, labels=[0, 1], zero_division=0)
    wp, wr, wf, _ = precision_recall_fscore_support(y_true, y_pred, average="weighted", zero_division=0)
    return {"accuracy": accuracy_score(y_true, y_pred),
            "nd_precision": pc[0], "nd_recall": rc[0], "nd_f1": fc[0],
            "precision": pc[1], "recall": rc[1], "f1": fc[1],
            "macro_f1": f1_score(y_true, y_pred, average="macro"),
            "weighted_precision": wp, "weighted_recall": wr, "weighted_f1": wf,
            "auroc": roc_auc_score(y_true, y_prob)}

def best_threshold(y_true, y_prob):
    ts = np.linspace(0.05, 0.95, 91)
    scores = [f1_score(y_true, (np.asarray(y_prob) >= t).astype(int), average="macro") for t in ts]
    return ts[int(np.argmax(scores))]

device: cuda
EATD 162 participants, 30 depressed
DAIC-WOZ 189 participants, 56 depressed


In [2]:
from sklearn.pipeline import Pipeline
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import RandomizedSearchCV, RepeatedStratifiedKFold, StratifiedKFold, cross_val_predict

pipe = Pipeline([
    ("tfidf", TfidfVectorizer()),
    ("clf", LogisticRegression(solver="liblinear", class_weight="balanced", max_iter=2000, random_state=42)),
])
shared = {
    "tfidf__min_df": [1, 2, 3], "tfidf__max_df": [0.8, 0.95, 1.0],
    "tfidf__sublinear_tf": [True, False], "tfidf__use_idf": [True, False],
    "clf__penalty": ["l1", "l2"], "clf__C": [0.01, 0.1, 1, 10, 100],
}
param_space = [
    {**shared, "tfidf__analyzer": ["word"], "tfidf__ngram_range": [(1, 1), (1, 2), (1, 3)],
     "tfidf__stop_words": [None, "english"], "tfidf__max_features": [None, 500, 2000]},
    {**shared, "tfidf__analyzer": ["char_wb"], "tfidf__ngram_range": [(2, 4), (3, 5)],
     "tfidf__max_features": [None, 5000]},
]

def tfidf_cross(src, tgt, n_iter=120):
    """Tune on the source dataset (train + dev, CV), then test on ALL target participants."""
    sd = src[src["split"].isin(["train", "dev"])]
    cv = RepeatedStratifiedKFold(n_splits=5, n_repeats=3, random_state=42)
    search = RandomizedSearchCV(pipe, param_space, n_iter=n_iter, scoring="roc_auc",
                                cv=cv, random_state=42, n_jobs=-1)
    search.fit(sd["text"], sd["label"])
    best = search.best_estimator_
    oof = cross_val_predict(best, sd["text"], sd["label"],
                            cv=StratifiedKFold(5, shuffle=True, random_state=42),
                            method="predict_proba")[:, 1]
    thr = best_threshold(sd["label"], oof)               # threshold chosen on SOURCE data only
    best.fit(sd["text"], sd["label"])
    prob = best.predict_proba(tgt["text"])[:, 1]
    return round(search.best_score_, 3), round(thr, 2), evaluate(tgt["label"], prob, thr)

rows = []
for src_name, tgt_name in [("EATD", "DAIC-WOZ"), ("DAIC-WOZ", "EATD")]:
    cv_auc, thr, m = tfidf_cross(DATA[src_name], DATA[tgt_name])
    rows.append({"train_on": src_name, "test_on": tgt_name, "model": "TF-IDF + LR",
                 "source_cv_auroc": cv_auc, "thr": thr, **{k: round(v, 3) for k, v in m.items()}})
    print(src_name, "->", tgt_name, "done")
tfidf_cross_results = pd.DataFrame(rows)
tfidf_cross_results.to_csv("cross_tfidf_results.csv", index=False)
tfidf_cross_results

EATD -> DAIC-WOZ done
DAIC-WOZ -> EATD done


,train_on,test_on,model,source_cv_auroc,thr,accuracy,nd_precision,nd_recall,nd_f1,precision,recall,f1,macro_f1,weighted_precision,weighted_recall,weighted_f1,auroc
0,EATD,DAIC-WOZ,TF-IDF + LR,0.645,0.09,0.513,0.703,0.534,0.607,0.295,0.464,0.361,0.484,0.582,0.513,0.534,0.496
1,DAIC-WOZ,EATD,TF-IDF + LR,0.769,0.36,0.765,0.818,0.917,0.864,0.214,0.100,0.136,0.500,0.706,0.765,0.729,0.495


In [3]:
def make_chunks(ids, size=510, stride=255):
    if len(ids) <= size:
        return [ids]
    chunks, start = [], 0
    while True:
        chunks.append(ids[start:start + size])
        if start + size >= len(ids):
            break
        start += stride
    return chunks

def build_samples(texts, labels, tokenizer, mode):
    cls_id = tokenizer.cls_token_id if tokenizer.cls_token_id is not None else tokenizer.bos_token_id
    sep_id = tokenizer.sep_token_id if tokenizer.sep_token_id is not None else tokenizer.eos_token_id
    samples = []
    for doc_id, (text, label) in enumerate(zip(texts, labels)):
        ids = tokenizer(text, add_special_tokens=False)["input_ids"]
        chunks = [ids[:510]] if mode == "base" else make_chunks(ids)
        for ch in chunks:
            samples.append(([cls_id] + ch + [sep_id], int(label), doc_id))
    return samples

def make_collate(tokenizer):
    pad_id = tokenizer.pad_token_id
    def collate(batch):
        max_len = max(len(b[0]) for b in batch)
        input_ids = torch.full((len(batch), max_len), pad_id, dtype=torch.long)
        attention_mask = torch.zeros((len(batch), max_len), dtype=torch.long)
        for i, b in enumerate(batch):
            input_ids[i, :len(b[0])] = torch.tensor(b[0])
            attention_mask[i, :len(b[0])] = 1
        return {"input_ids": input_ids, "attention_mask": attention_mask,
                "labels": torch.tensor([b[1] for b in batch])}
    return collate

def predict(model, samples, n_docs, collate, batch_size=8):
    model.eval()
    probs = []
    with torch.no_grad():
        for batch in DataLoader(samples, batch_size=batch_size, shuffle=False, collate_fn=collate):
            inputs = {k: v.to(device) for k, v in batch.items() if k != "labels"}
            probs += torch.softmax(model(**inputs).logits, dim=-1)[:, 1].cpu().tolist()
    doc_ids = np.array([s[2] for s in samples])
    return np.bincount(doc_ids, weights=probs, minlength=n_docs) / np.bincount(doc_ids, minlength=n_docs)

def train_cross(model_name, mode, lr, src, tgt, epochs=10, patience=3, batch_size=4, seed=42):
    """Train on source train (early stopping on source dev), predict ALL target participants."""
    set_seed(seed)
    s_train = src[src["split"] == "train"].reset_index(drop=True)
    s_dev = src[src["split"] == "dev"].reset_index(drop=True)
    tgt = tgt.reset_index(drop=True)

    tokenizer = AutoTokenizer.from_pretrained(model_name)
    collate = make_collate(tokenizer)
    tr = build_samples(s_train["text"], s_train["label"], tokenizer, mode)
    dv = build_samples(s_dev["text"], s_dev["label"], tokenizer, mode)
    tg = build_samples(tgt["text"], tgt["label"], tokenizer, mode)

    model = AutoModelForSequenceClassification.from_pretrained(model_name, num_labels=2).to(device)
    counts = np.bincount(s_train["label"])
    class_w = torch.tensor(len(s_train) / (2 * counts), dtype=torch.float).to(device)
    loss_fn = nn.CrossEntropyLoss(weight=class_w)

    loader = DataLoader(tr, batch_size=batch_size, shuffle=True, collate_fn=collate)
    opt = torch.optim.AdamW(model.parameters(), lr=lr, weight_decay=0.01)
    total = len(loader) * epochs
    sched = get_linear_schedule_with_warmup(opt, int(0.1 * total), total)

    best_auc, best_state, bad = -1, None, 0
    for epoch in range(epochs):
        model.train()
        for batch in loader:
            labels = batch["labels"].to(device)
            inputs = {k: v.to(device) for k, v in batch.items() if k != "labels"}
            loss = loss_fn(model(**inputs).logits, labels)
            loss.backward()
            torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
            opt.step(); sched.step(); opt.zero_grad()
        auc = roc_auc_score(s_dev["label"], predict(model, dv, len(s_dev), collate))
        if auc > best_auc:
            best_auc, bad = auc, 0
            best_state = {k: v.detach().cpu().clone() for k, v in model.state_dict().items()}
        else:
            bad += 1
            if bad >= patience:
                break

    model.load_state_dict(best_state)
    dev_prob = predict(model, dv, len(s_dev), collate)
    tgt_prob = predict(model, tg, len(tgt), collate)
    del model, opt; gc.collect(); torch.cuda.empty_cache()
    return s_dev, dev_prob, tgt_prob

In [4]:
MODELS = {"BERT": "bert-base-uncased", "RoBERTa": "roberta-base"}
MODES = ["base", "window"]
SEEDS = [0, 1, 2, 3, 4]
BATCH = {"EATD": 8, "DAIC-WOZ": 4}                       # batch size by SOURCE dataset
LR_FILES = {"EATD": "eatd_transformer_results.csv", "DAIC-WOZ": "daicwoz_transformer_results.csv"}

runs = []
for src_name, tgt_name in [("EATD", "DAIC-WOZ"), ("DAIC-WOZ", "EATD")]:
    if os.path.exists(LR_FILES[src_name]):
        best_lr = {r.model: r.lr for r in pd.read_csv(LR_FILES[src_name]).itertuples()}
    else:
        print(f"{LR_FILES[src_name]} not found, using lr=2e-5 for all")
        best_lr = {f"{n}-{m}": 2e-5 for n in MODELS for m in MODES}
    for name, hf_name in MODELS.items():
        for mode in MODES:
            key = f"{name}-{mode}"
            for seed in SEEDS:
                t0 = time.time()
                s_dev, dev_prob, tgt_prob = train_cross(hf_name, mode, best_lr[key], DATA[src_name], DATA[tgt_name],
                                                        batch_size=BATCH[src_name], seed=seed)
                thr = best_threshold(s_dev["label"], dev_prob)      # threshold from SOURCE dev only
                m = evaluate(DATA[tgt_name]["label"], tgt_prob, thr)
                runs.append({"train_on": src_name, "test_on": tgt_name, "model": key, "seed": seed, **m})
                print(f"{src_name}->{tgt_name} {key} seed={seed}: AUROC={m['auroc']:.3f} "
                      f"macro_f1={m['macro_f1']:.3f} ({time.time() - t0:.0f}s)")

runs_df = pd.DataFrame(runs)
runs_df.to_csv("cross_transformer_runs.csv", index=False)

metrics = ["accuracy", "nd_precision", "nd_recall", "nd_f1", "precision", "recall", "f1",
           "macro_f1", "weighted_precision", "weighted_recall", "weighted_f1", "auroc"]
g = runs_df.groupby(["train_on", "test_on", "model"], sort=False)[metrics]
summary = g.mean().round(3).astype(str) + " ± " + g.std().round(3).astype(str)
summary

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

EATD->DAIC-WOZ BERT-base seed=0: AUROC=0.448 macro_f1=0.413 (13s)


Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

EATD->DAIC-WOZ BERT-base seed=1: AUROC=0.486 macro_f1=0.479 (7s)


Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

EATD->DAIC-WOZ BERT-base seed=2: AUROC=0.502 macro_f1=0.413 (11s)


... (71 outputs left out)

DAIC-WOZ->EATD RoBERTa-window seed=3: AUROC=0.558 macro_f1=0.274 (90s)


Loading weights:   0%|          | 0/197 [00:00<?, ?it/s]

DAIC-WOZ->EATD RoBERTa-window seed=4: AUROC=0.519 macro_f1=0.504 (78s)


accuracy   nd_precision      nd_recall  \
train_on test_on  model                                                         
EATD     DAIC-WOZ BERT-base       0.697 ± 0.014  0.704 ± 0.002   0.982 ± 0.04   
                  BERT-window      0.618 ± 0.18  0.562 ± 0.314  0.794 ± 0.444   
                  RoBERTa-base    0.684 ± 0.032   0.71 ± 0.013  0.932 ± 0.078   
                  RoBERTa-window  0.561 ± 0.184  0.597 ± 0.335  0.588 ± 0.426   
DAIC-WOZ EATD     BERT-base       0.362 ± 0.263  0.473 ± 0.434  0.283 ± 0.417   
                  BERT-window     0.414 ± 0.206  0.876 ± 0.089  0.358 ± 0.331   
                  RoBERTa-base    0.194 ± 0.016  0.375 ± 0.515  0.012 ± 0.023   
                  RoBERTa-window  0.435 ± 0.277   0.88 ± 0.076  0.389 ± 0.446   

                                          nd_f1      precision         recall  \
train_on test_on  model                                                         
EATD     DAIC-WOZ BERT-base        0.82 ± 0.013  0.067 ± 0.149  0.021 ± 0.048   
                  BERT-window     0.658 ± 0.368  0.059 ± 0.133    0.2 ± 0.447   
                  RoBERTa-base    0.805 ± 0.029  0.216 ± 0.215  0.093 ± 0.105   
                  RoBERTa-window  0.562 ± 0.357  0.402 ± 0.091  0.496 ± 0.406   
DAIC-WOZ EATD     BERT-base       0.306 ± 0.383  0.147 ± 0.082  0.707 ± 0.418   
                  BERT-window     0.426 ± 0.322   0.19 ± 0.009   0.66 ± 0.346   
                  RoBERTa-base    0.023 ± 0.044  0.186 ± 0.001  0.993 ± 0.015   
                  RoBERTa-window  0.411 ± 0.395  0.154 ± 0.086  0.633 ± 0.467   

                                             f1       macro_f1  \
train_on test_on  model                                          
EATD     DAIC-WOZ BERT-base       0.032 ± 0.073   0.426 ± 0.03   
                  BERT-window     0.091 ± 0.204  0.375 ± 0.082   
                  RoBERTa-base    0.126 ± 0.138  0.465 ± 0.061   
                  RoBERTa-window   0.34 ± 0.182  0.451 ± 0.146   
DAIC-WOZ EATD     BERT-base       0.243 ± 0.136  0.274 ± 0.136   
                  BERT-window     0.276 ± 0.058  0.351 ± 0.136   
                  RoBERTa-base    0.313 ± 0.001  0.168 ± 0.022   
                  RoBERTa-window  0.237 ± 0.138  0.324 ± 0.143   

                                 weighted_precision weighted_recall  \
train_on test_on  model                                               
EATD     DAIC-WOZ BERT-base           0.516 ± 0.045   0.697 ± 0.014   
                  BERT-window         0.413 ± 0.182    0.618 ± 0.18   
                  RoBERTa-base        0.563 ± 0.071   0.684 ± 0.032   
                  RoBERTa-window      0.539 ± 0.253   0.561 ± 0.184   
DAIC-WOZ EATD     BERT-base           0.413 ± 0.347   0.362 ± 0.263   
                  BERT-window         0.749 ± 0.073   0.414 ± 0.206   
                  RoBERTa-base          0.34 ± 0.42   0.194 ± 0.016   
                  RoBERTa-window      0.746 ± 0.071   0.435 ± 0.277   

                                    weighted_f1          auroc  
train_on test_on  model                                         
EATD     DAIC-WOZ BERT-base       0.587 ± 0.012  0.474 ± 0.022  
                  BERT-window      0.49 ± 0.198   0.49 ± 0.059  
                  RoBERTa-base    0.604 ± 0.033  0.564 ± 0.041  
                  RoBERTa-window  0.496 ± 0.224  0.553 ± 0.036  
DAIC-WOZ EATD     BERT-base        0.294 ± 0.29  0.491 ± 0.054  
                  BERT-window     0.399 ± 0.253  0.511 ± 0.051  
                  RoBERTa-base    0.077 ± 0.036  0.504 ± 0.034  
                  RoBERTa-window    0.379 ± 0.3   0.538 ± 0.02